# GPU-Accelerated DBSCAN Clustering

This notebook demonstrates the GPU-accelerated DBSCAN implementation and compares it with the CPU version.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from ghdbscan import DBSCAN, DBSCAN_GPU

# Set random seed for reproducibility
np.random.seed(42)

## 1. Check GPU Availability

In [ ]:
gpu_available = DBSCAN_GPU.is_available()
print(f"GPU Available: {gpu_available}")

if not gpu_available:
    print("⚠️  No GPU detected. The GPU implementation will fall back to CPU for small datasets.")

## 2. Generate Sample Data

Create a dataset with three distinct clusters and some noise points.

In [ ]:
# Create clusters
n_samples = 1000
cluster1 = np.random.randn(n_samples, 2) * 0.5 + [0, 0]
cluster2 = np.random.randn(n_samples, 2) * 0.7 + [5, 5]
cluster3 = np.random.randn(n_samples, 2) * 0.6 + [10, 2]
noise = np.random.uniform(-2, 12, (100, 2))

X = np.vstack([cluster1, cluster2, cluster3, noise])

print(f"Dataset shape: {X.shape}")
print(f"Total points: {len(X)}")

## 3. Visualize the Data

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], alpha=0.5, s=20)
plt.title('Original Data')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)
plt.show()

## 4. CPU DBSCAN Clustering

In [ ]:
# Run CPU DBSCAN
print("Running CPU DBSCAN...")
start_time = time.time()

dbscan_cpu = DBSCAN(eps=0.8, min_samples=10)
labels_cpu = dbscan_cpu.fit_predict(X)

cpu_time = time.time() - start_time

n_clusters_cpu = len(set(labels_cpu)) - (1 if -1 in labels_cpu else 0)
n_noise_cpu = list(labels_cpu).count(-1)

print(f"CPU Time: {cpu_time:.4f} seconds")
print(f"Clusters found: {n_clusters_cpu}")
print(f"Noise points: {n_noise_cpu}")

## 5. GPU DBSCAN Clustering

In [ ]:
# Run GPU DBSCAN
print("Running GPU DBSCAN...")
start_time = time.time()

dbscan_gpu = DBSCAN_GPU(eps=0.8, min_samples=10)
labels_gpu = dbscan_gpu.fit_predict(X)

gpu_time = time.time() - start_time

n_clusters_gpu = len(set(labels_gpu)) - (1 if -1 in labels_gpu else 0)
n_noise_gpu = list(labels_gpu).count(-1)

print(f"GPU Time: {gpu_time:.4f} seconds")
print(f"Clusters found: {n_clusters_gpu}")
print(f"Noise points: {n_noise_gpu}")

## 6. Performance Comparison

In [ ]:
speedup = cpu_time / gpu_time if gpu_time > 0 else 0

print(f"\n{'='*50}")
print("Performance Comparison")
print(f"{'='*50}")
print(f"CPU Time:  {cpu_time:.4f}s")
print(f"GPU Time:  {gpu_time:.4f}s")
print(f"Speedup:   {speedup:.2f}x")
print(f"Results Match: {np.array_equal(labels_cpu, labels_gpu)}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
ax1.bar(['CPU', 'GPU'], [cpu_time, gpu_time], color=['#3498db', '#2ecc71'])
ax1.set_ylabel('Time (seconds)')
ax1.set_title('Execution Time Comparison')
ax1.grid(True, alpha=0.3)

# Add speedup text
ax1.text(1, gpu_time, f'{speedup:.2f}x faster', 
         ha='center', va='bottom', fontsize=12, fontweight='bold')

# Speedup bar
ax2.barh(['Speedup'], [speedup], color='#e74c3c')
ax2.set_xlabel('Speedup Factor')
ax2.set_title('GPU Speedup')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Visualize Clustering Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# CPU results
unique_labels_cpu = set(labels_cpu)
colors_cpu = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels_cpu))]

for k, col in zip(unique_labels_cpu, colors_cpu):
    if k == -1:
        col = [0, 0, 0, 1]  # Black for noise
    
    class_member_mask = (labels_cpu == k)
    xy = X[class_member_mask]
    ax1.scatter(xy[:, 0], xy[:, 1], c=[col], s=20, alpha=0.6,
               label=f'Cluster {k}' if k != -1 else 'Noise')

ax1.set_title(f'CPU DBSCAN Results\n{n_clusters_cpu} clusters, {n_noise_cpu} noise points')
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
ax1.grid(True, alpha=0.3)

# GPU results
unique_labels_gpu = set(labels_gpu)
colors_gpu = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels_gpu))]

for k, col in zip(unique_labels_gpu, colors_gpu):
    if k == -1:
        col = [0, 0, 0, 1]  # Black for noise
    
    class_member_mask = (labels_gpu == k)
    xy = X[class_member_mask]
    ax2.scatter(xy[:, 0], xy[:, 1], c=[col], s=20, alpha=0.6,
               label=f'Cluster {k}' if k != -1 else 'Noise')

ax2.set_title(f'GPU DBSCAN Results\n{n_clusters_gpu} clusters, {n_noise_gpu} noise points')
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Scaling Test

Test performance with different dataset sizes.

In [ ]:
sizes = [500, 1000, 2000, 5000, 10000]
cpu_times = []
gpu_times = []

for size in sizes:
    print(f"Testing with {size} points...")
    
    # Generate data
    X_test = np.vstack([
        np.random.randn(size // 2, 2) + [0, 0],
        np.random.randn(size // 2, 2) + [5, 5]
    ])
    
    # CPU
    start = time.time()
    dbscan_cpu = DBSCAN(eps=0.8, min_samples=5)
    _ = dbscan_cpu.fit_predict(X_test)
    cpu_times.append(time.time() - start)
    
    # GPU
    start = time.time()
    dbscan_gpu = DBSCAN_GPU(eps=0.8, min_samples=5)
    _ = dbscan_gpu.fit_predict(X_test)
    gpu_times.append(time.time() - start)

# Plot scaling
plt.figure(figsize=(12, 6))
plt.plot(sizes, cpu_times, 'o-', label='CPU', linewidth=2, markersize=8)
plt.plot(sizes, gpu_times, 's-', label='GPU', linewidth=2, markersize=8)
plt.xlabel('Dataset Size (points)')
plt.ylabel('Time (seconds)')
plt.title('DBSCAN Performance Scaling')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Speedup plot
speedups = [cpu / gpu if gpu > 0 else 0 for cpu, gpu in zip(cpu_times, gpu_times)]
plt.figure(figsize=(12, 6))
plt.plot(sizes, speedups, 'o-', color='#e74c3c', linewidth=2, markersize=8)
plt.axhline(y=1, color='gray', linestyle='--', label='No speedup')
plt.xlabel('Dataset Size (points)')
plt.ylabel('Speedup Factor')
plt.title('GPU Speedup vs Dataset Size')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrated:
- GPU-accelerated DBSCAN clustering
- Performance comparison with CPU implementation
- Scaling behavior with different dataset sizes

**Key Takeaways:**
- GPU acceleration provides significant speedup for large datasets
- Small datasets may see overhead from GPU initialization
- Results are identical between CPU and GPU implementations